# *Introduction*


This Notebook consists of:
1. txt to csv data conversion
2. data preprocessing (handle null & duplicated values, text cleaning, remove stopwords, shortwords)
3. Lemmetization
4. Doc2Vec
5. K-Means with various K
6. Silhuette score
7. conclusion

# *Import Necessary Dependencies*

In [1]:
!pip install nltk

import numpy as np
import pandas as pd
import os

# Data Preprocessing
import nltk
import re
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

# Vectorization
!pip install gensim
import gensim
from gensim.models.doc2vec import Doc2Vec

#k-means
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#ignore warnings
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package wordnet to /root/nltk_data...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 15.6 MB/s eta 0:00:00


# Download Data

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamkushwaha/bbc-full-text-document-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bbc-full-text-document-classification' dataset.
Path to dataset files: /kaggle/input/bbc-full-text-document-classification


# 1. convert text data to csv

In [3]:
import os
for dirname in os.walk('/kaggle/input/bbc-full-text-document-classification'):
        print(dirname)

('/kaggle/input/bbc-full-text-document-classification', ['bbc-fulltext (document classification)', 'bbc'], [])
('/kaggle/input/bbc-full-text-document-classification/bbc-fulltext (document classification)', ['bbc'], [])
('/kaggle/input/bbc-full-text-document-classification/bbc-fulltext (document classification)/bbc', ['politics', 'sport', 'tech', 'entertainment', 'business'], ['README.TXT'])
('/kaggle/input/bbc-full-text-document-classification/bbc-fulltext (document classification)/bbc/politics', [], ['361.txt', '245.txt', '141.txt', '372.txt', '333.txt', '276.txt', '244.txt', '175.txt', '351.txt', '265.txt', '178.txt', '201.txt', '087.txt', '251.txt', '122.txt', '375.txt', '098.txt', '139.txt', '161.txt', '328.txt', '174.txt', '353.txt', '383.txt', '409.txt', '039.txt', '365.txt', '028.txt', '134.txt', '044.txt', '154.txt', '054.txt', '200.txt', '396.txt', '021.txt', '179.txt', '229.txt', '322.txt', '057.txt', '043.txt', '045.txt', '148.txt', '115.txt', '415.txt', '040.txt', '289.txt'

In [4]:
DATASET_PATH = '/kaggle/input/bbc-full-text-document-classification/bbc'

rows = []

for category in os.listdir(DATASET_PATH):

    category_path = os.path.join(DATASET_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for file in os.listdir(category_path):

        if not file.endswith(".txt"):
            continue

        file_path = os.path.join(category_path, file)

        try:
            with open(file_path, "r", encoding="latin-1") as f:
                text = f.read()

            rows.append({
                "category": category,
                "filename": file,
                "text": text
            })

        except Exception as e:
            print(f"Error reading {file}: {e}")

df = pd.DataFrame(rows)

print(df.shape)

df.to_csv(
    "bbc_news_dataset.csv",
    index=False
)

print("Saved: bbc_news_dataset.csv")

(2225, 3)
Saved: bbc_news_dataset.csv


In [5]:
news_data = pd.read_csv("bbc_news_dataset.csv")

# 2. Data preprocessing

In [6]:
news_data.head()

,category,filename,text
0,politics,361.txt,Budget to set scene for election\n\nGordon Bro...
1,politics,245.txt,Army chiefs in regiments decision\n\nMilitary ...
2,politics,141.txt,Howard denies split over ID cards\n\nMichael H...
3,politics,372.txt,Observers to monitor UK election\n\nMinisters ...
4,politics,333.txt,Kilroy names election seat target\n\nEx-chat s...


In [7]:
news_data.category.isnull().sum()

np.int64(0)

In [8]:
news_data.filename.isnull().sum()

np.int64(0)

In [9]:
if "readme" or "README" in news_data.filename :
  print("yes")
else :
  print("no")

yes


In [10]:
# drop rows containing readme.txt in filename
news_data = news_data[news_data.filename != "README.TXT"]

In [11]:
news_data[news_data.filename == "README.TXT"].count()

,0
category,0
filename,0
text,0


In [12]:
news_data.text.isnull().sum()

np.int64(0)

### text cleaning

In [13]:
def clean_text(text):
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]*>', '', text)

    # 2. Remove URLs / Hyperlinks
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Remove Email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # 4. Remove everything except letters and numbers (keeps spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # 5. Remove numbers/digits
    text = re.sub(r'\d+', '', text)

    # 6. Replace multiple spaces/newlines with a single space
    text = re.sub(r'\s+', ' ', text)

    # 7. Strip leading and trailing whitespace and lowercase
    return text.strip().lower()

In [14]:
news_data['clean_text'] = news_data['text'].apply(lambda x: clean_text(x))

In [15]:
news_data.head(1)['clean_text'].tolist()

['budget to set scene for election gordon brown will seek to put the economy at the centre of labours bid for a third term in power when he delivers his ninth budget at gmt he is expected to stress the importance of continued economic stability with low unemployment and interest rates the chancellor is expected to freeze petrol duty and raise the stamp duty threshold from but the conservatives and lib dems insist voters face higher taxes and more meanstesting under labour treasury officials have said there will not be a preelection giveaway but mr brown is thought to have about bn to spare increase in the stamp duty threshold from a freeze on petrol duty an extension of tax credit scheme for poorer families possible help for pensioners the stamp duty threshold rise is intended to help first time buyers a likely theme of all three of the main parties general election manifestos ten years ago buyers had a much greater chance of avoiding stamp duty with close to half a million properties 

### remove stopwords

In [16]:
stop_words = set(stopwords.words('english'))

In [17]:
custom_stopwords = {
    "said", "would", "could", "also",
    "say", "told", "one", "new",
    "year", "years", "week",
    "last", "next", "may",
    "make", "made", "added",
    "bbc", "people", "time"
}

In [18]:
news_data['clean_text']= news_data['clean_text'].apply(lambda x: ' '.join(word for word in x.split() if word not in stop_words))

In [19]:
news_data['clean_text'] = news_data['clean_text'].apply(lambda x: ' '.join(word for word in x.split() if word not in custom_stopwords))

### removing short words

In [20]:
news_data['important_text'] = news_data['clean_text'].apply(lambda x: ' '.join(word for word in x.split() if len(word) > 2))

In [21]:
news_data['important_text'].head(1).tolist()

['budget set scene election gordon brown seek put economy centre labours bid third term power delivers ninth budget gmt expected stress importance continued economic stability low unemployment interest rates chancellor expected freeze petrol duty raise stamp duty threshold conservatives lib dems insist voters face higher taxes meanstesting labour treasury officials preelection giveaway brown thought spare increase stamp duty threshold freeze petrol duty extension tax credit scheme poorer families possible help pensioners stamp duty threshold rise intended help first buyers likely theme three main parties general election manifestos ten ago buyers much greater chance avoiding stamp duty close half million properties england wales alone selling less since average property prices doubled starting threshold stamp duty increased tax credits result number properties incurring stamp duty rocketed governments tax take liberal democrats unveiled proposals raise stamp duty threshold february tor

# 3. Lemmetization

In [23]:
lemmatizer = WordNetLemmatizer()

news_data['lemme'] = news_data['important_text'].apply(
    lambda x: ' '.join(
        lemmatizer.lemmatize(word)
        for word in x.split()
    )
)

# 4. Doc2Vec

In [26]:
from gensim.models.doc2vec import TaggedDocument

documents = [
    TaggedDocument(
        words=text.split(),
        tags=[str(i)]
    )
    for i, text in enumerate(news_data["lemme"])
]

In [30]:
model = Doc2Vec(
    documents,
    vector_size=100,
    window=5,
    min_count=3,
    dm=1,
    epochs=30,
    workers=4
)

In [31]:
doc_vectors = np.array([
    model.dv[str(i)]
    for i in range(len(documents))
])

In [32]:
print(doc_vectors.shape)

(2225, 100)


### finding cosine similarity

In [33]:
# finding cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [doc_vectors[1]],
    doc_vectors
)

In [35]:
similarities.argsort()[0][-10:]

array([171,  81, 261,   9, 296, 189,  56, 126, 329,   1])

In [38]:
list_last_10 = similarities.argsort()[0][-10:]

for i in list_last_10:
    print(f"{i} : Domain: {news_data.iloc[i]['category']} & {news_data.iloc[i]['text']}")

171 : Domain: politics & Police chief backs drinking move

A chief constable has backed the introduction of 24-drinking, saying police had a responsibility to ensure people could benefit from a law change.

However, Norfolk police chief Andy Hayman also warned that a great deal of preparatory work was still needed. "I don't subscribe to the views of some of my colleagues who are coming out and objecting to it," he said. His comments come after the Liberal Democrats backed Tory demands that the government's plans be put on hold. Andy Hayman said he did not agree with politicians and senior police officers who have objected to the plans, which come into force on 7 February. "I feel that is a premature position to be taking," he said. Among those who have criticised the plans are the UK's top policeman Sir John Stevens.

The Metropolitan police chief said last week that the plans for 24-hour drinking should be re-examined because of a binge drinking "epidemic". However, Mr Hayman said: "I

sounds well distributed

# 5,6. K-means & Silhuette score

In [43]:
for k in range(2, 11):
  k_means = KMeans(
      n_clusters = k,
      random_state = 42,
      n_init = 10
  )

  labels = k_means.fit_predict(doc_vectors)

  score = silhouette_score(doc_vectors, labels)
  print(f"Silhouette Score for k={k}: {score}")

Silhouette Score for k=2: 0.06764724850654602
Silhouette Score for k=3: 0.06705082207918167
Silhouette Score for k=4: 0.05758701264858246
Silhouette Score for k=5: 0.052891504019498825
Silhouette Score for k=6: 0.05591220036149025
Silhouette Score for k=7: 0.0631881058216095
Silhouette Score for k=8: 0.04562464728951454
Silhouette Score for k=9: 0.049535948783159256
Silhouette Score for k=10: 0.044707100838422775


# 7. Conclusion

On the BBC News corpus (~2200 documents), Word2Vec with mean-pooled document embeddings substantially outperformed Doc2Vec for unsupervised clustering.